In [55]:
##### Evaluate RF models 
# calculates model performance metrics 

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [56]:
##### SET-UP

# Get the current working directory
cd = Path.cwd().parent.parent 

# import reference data 
ref_data = pd.read_csv(f"{cd}/Data/Clean/Intensities/intensities_for_eval.csv")

# import model predictions
capital_rf = pd.read_parquet(f'{cd}/Results/RF_models_final/capital_rf_spatial_CV/predictions.parquet')
labor_rf = pd.read_parquet(f'{cd}/Results/RF_models_final/labor_rf_spatial_CV/predictions.parquet')

capital_qrf = pd.read_parquet(f'{cd}/Results/RF_models_final/capital_qrf_spatial_CV/predictions.parquet')
labor_qrf = pd.read_parquet(f'{cd}/Results/RF_models_final/labor_qrf_spatial_CV/predictions.parquet')

In [57]:
##### PREP DATA

# Add original intensity data (sub-national and country) for evaluation
capital_rf = capital_rf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
labor_rf = labor_rf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
capital_qrf = capital_qrf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
labor_qrf = labor_qrf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')

# Estimate actual predicted intensity (not relative)
capital_rf['predicted_log_capital_intensity_USD_per_million_tonne'] = capital_rf['prediction'] + capital_rf['log_country_capital_intensity_USD_per_million_tonne']
capital_rf['predicted_capital_intensity_USD_per_million_tonne'] = np.expm1(capital_rf['predicted_log_capital_intensity_USD_per_million_tonne'])
capital_qrf['q10_predicted_log_capital_intensity_USD_per_million_tonne'] = capital_qrf['q10'] + capital_qrf['log_country_capital_intensity_USD_per_million_tonne']
capital_qrf['q90_predicted_log_capital_intensity_USD_per_million_tonne'] = capital_qrf['q90'] + capital_qrf['log_country_capital_intensity_USD_per_million_tonne']

labor_rf['predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_rf['prediction'] + labor_rf['log_country_labor_intensity_jobs_per_million_tonne']
labor_rf['predicted_labor_intensity_jobs_per_million_tonne'] = np.expm1(labor_rf['predicted_log_labor_intensity_jobs_per_million_tonne'])
labor_qrf['q10_predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_qrf['q10'] + labor_qrf['log_country_labor_intensity_jobs_per_million_tonne']
labor_qrf['q90_predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_qrf['q90'] + labor_qrf['log_country_labor_intensity_jobs_per_million_tonne']

In [58]:
##### DEFINE FUNCTIONS

# function to compute key performance metrics per fold 
def compute_metrics_by_fold(df, target_col, pred_col):
    rows = []
    for fold, g in df.groupby("fold"):
        y_true = g[target_col]
        y_pred = g[pred_col]
        resid  = y_true - y_pred

        rows.append({
            "fold":      fold,
            "n":         len(g),
            "R2":        r2_score(y_true, y_pred),
            "RMSE":      np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAE":       mean_absolute_error(y_true, y_pred),
            "Bias":      resid.mean(),
            "Pearson_r": y_true.corr(y_pred),
        })

    metrics_df = pd.DataFrame(rows).sort_values("fold").reset_index(drop=True)

    # summary rows across folds (unweighted — averaged, not pooled)
    metric_cols = ["R2", "RMSE", "MAE", "Bias", "Pearson_r"]
    mean_row = {"fold": "mean", "n": metrics_df["n"].sum()}
    var_row  = {"fold": "variance", "n": np.nan}
    for col in metric_cols:
        mean_row[col] = metrics_df[col].mean()
        var_row[col]  = metrics_df[col].var()  

    summary = pd.DataFrame([mean_row, var_row])
    return pd.concat([metrics_df, summary], ignore_index=True)


# function to evaluate prediction intervals from QRF, by fold
def compute_interval_metrics_by_fold(df, target_col, q10col, q90col):
    rows = []
    for fold, g in df.groupby("fold"):
        y_true = g[target_col]
        low    = g[q10col]
        high   = g[q90col]
        covered   = (y_true >= low) & (y_true <= high)

        rows.append({
            "fold":             fold,
            "n":                len(g),
            "coverage":         covered.mean(),
            "nominal_coverage": 0.8,
            "mean_width":       (high - low).mean(),
        })

    metrics_df = pd.DataFrame(rows).sort_values("fold").reset_index(drop=True)

    # summary rows across folds (unweighted -- averaged, not pooled)
    metric_cols = ["coverage", "mean_width"]
    mean_row = {"fold": "mean", "n": metrics_df["n"].sum(), "nominal_coverage": 0.8}
    var_row  = {"fold": "variance", "n": np.nan, "nominal_coverage": np.nan}
    for col in metric_cols:
        mean_row[col] = metrics_df[col].mean()
        var_row[col]  = metrics_df[col].var()

    summary = pd.DataFrame([mean_row, var_row])
    return pd.concat([metrics_df, summary], ignore_index=True)

In [59]:
##### CAPITAL MODEL METRICS

### prep data ---------------------------------------------------------------------------

# set target column
target_col = "rtv_log_capital_intensity_USD_per_million_tonne" 

# set output directory
out_dir = Path(f'{cd}/Results/RF_models_final/capital_rf_spatial_CV')
out_dir_qrf = Path(f'{cd}/Results/RF_models_final/capital_qrf_spatial_CV')

# split into test and train
test_df  = capital_rf[capital_rf["split"] == "test"]
train_df = capital_rf[capital_rf["split"] == "train"]
test_df_qrf = capital_qrf[capital_qrf["split"] == "test"]

# calculate metrics ----------------------------------------------------------------------

print("\n── PREDICTED RELATIVE INTENSITIES (scale: relative log) ─────────────────────────────")
# measures of model performance, not real work performance 
# training data: how well does the model perform on data it knows
train_metrics = compute_metrics_by_fold(train_df, target_col, 'prediction')
print("\n── TRAIN ──")
print(train_metrics.to_string(index=False))
# test data: how well does the model perform on data it doesn't know
test_metrics = compute_metrics_by_fold(test_df, target_col, 'prediction')
print("\n── TEST ──")
print(test_metrics.to_string(index=False))
# test data, but using median QRF model
qrf_median_metrics = compute_metrics_by_fold(test_df_qrf, target_col, "q50")
print("\n── TEST: QRF Q50 ──")
print(qrf_median_metrics.to_string(index=False))
# measure of QRF interval performance 
interval_metrics = compute_interval_metrics_by_fold(test_df_qrf, target_col, 'q10', 'q90')
print("\n── TEST: QRF interval ──")
print(interval_metrics.to_string(index=False))

# measures of model on actual variable of interest 
# log scale is more appropriate because of right-tail distribution in underlying data 
# can calculate on original scale, but would be heavily skewed by large outliers 
print("\n── PREDICTED ACTUAL INTENSITIES (scale: log) ────────────────────────────────────────")
convt_log_metrics = compute_metrics_by_fold(test_df, 'log_region_capital_intensity_USD_per_million_tonne', 'predicted_log_capital_intensity_USD_per_million_tonne')
print("\n── TEST ──")
print(convt_log_metrics.to_string(index=False))
# QRF interval  
interval_metrics = compute_interval_metrics_by_fold(test_df_qrf, 'log_region_capital_intensity_USD_per_million_tonne', 'q10_predicted_log_capital_intensity_USD_per_million_tonne', 'q90_predicted_log_capital_intensity_USD_per_million_tonne')
print("\n── TEST: QRF interval ──")
print(interval_metrics.to_string(index=False))

# measures of naive country average model on actual variable of interest 
# log scale is more appropriate because of right-tail distribution in underlying data 
# can calculate on original scale, but would be heavily skewed by large outliers 
print("\n── COUNTRY AVERAGE MODEL (scale: log) ────────────────----------───────-─────────────")
country_log_metrics = compute_metrics_by_fold(test_df, 'log_region_capital_intensity_USD_per_million_tonne', 'log_country_capital_intensity_USD_per_million_tonne')
print("\n── TEST ──")
print(country_log_metrics.to_string(index=False))

### save metrics -------------------------------------------------------------------------

# output directory
out_dir = Path(f'{cd}/Results/RF_models_final/labor_rf_spatial_CV')
out_dir_qrf = Path(f'{cd}/Results/RF_models_final/labor_qrf_spatial_CV')

# relative log metrics (train + test combined)
train_metrics["split"] = "train"
test_metrics["split"]  = "test"
relative_metrics = pd.concat([train_metrics, test_metrics], ignore_index=True)
relative_metrics.to_csv(out_dir / "relative_log_metrics.csv", index=False)

# actual intensities, log scale (test only)
convt_log_metrics["split"] = "test"
convt_log_metrics.to_csv(out_dir / "actual_log_metrics.csv", index=False)

# country average baseline, log scale (test only)
country_log_metrics["split"] = "test"
country_log_metrics.to_csv(out_dir / "country_avg_log_metrics.csv", index=False)

# QRF interval metrics (coverage / width, test only)
interval_metrics["split"] = "test"
interval_metrics.to_csv(out_dir_qrf / "qrf_interval_metrics.csv", index=False)

# QRF median (q50) point-prediction metrics, relative log scale (test only)
qrf_median_metrics["split"] = "test"
qrf_median_metrics.to_csv(out_dir_qrf / "qrf_median_metrics.csv", index=False)


── PREDICTED RELATIVE INTENSITIES (scale: relative log) ─────────────────────────────

── TRAIN ──
    fold       n       R2     RMSE      MAE     Bias  Pearson_r
  fold_1  1873.0 0.838602 0.378083 0.271930 0.008155   0.928804
  fold_2  1859.0 0.873831 0.540594 0.404360 0.009362   0.944994
  fold_3  2646.0 0.888005 0.453395 0.329270 0.007623   0.950225
  fold_4  2650.0 0.890614 0.443263 0.323054 0.009894   0.952128
  fold_5  2652.0 0.890291 0.447930 0.326568 0.001880   0.952103
    mean 11680.0 0.876269 0.452653 0.331036 0.007383   0.945651
variance     NaN 0.000491 0.003351 0.002239 0.000010   0.000097

── TEST ──
    fold      n        R2     RMSE      MAE      Bias  Pearson_r
  fold_1 1047.0  0.426934 1.317159 1.113089  0.295782   0.768674
  fold_2 1061.0  0.491779 0.620442 0.481340  0.131366   0.718101
  fold_3  274.0  0.195448 0.757555 0.595489  0.161293   0.553518
  fold_4  270.0  0.545478 0.803727 0.594069  0.108366   0.744148
  fold_5  268.0 -0.230761 0.910817 0.682652 -0.3579

/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [60]:
##### LABOR MODEL METRICS

### prep data ---------------------------------------------------------------------------

# set target column
target_col = "rtv_log_labor_intensity_jobs_per_million_tonne" 

# set output directory
out_dir = Path(f'{cd}/Results/RF_models_final/labor_rf_spatial_CV')
out_dir_qrf = Path(f'{cd}/Results/RF_models_final/labor_qrf_spatial_CV')

# split into test and train
test_df  = labor_rf[labor_rf["split"] == "test"]
train_df = labor_rf[labor_rf["split"] == "train"]
test_df_qrf = labor_qrf[labor_qrf["split"] == "test"]

# calculate metrics ----------------------------------------------------------------------

print("\n── PREDICTED RELATIVE INTENSITIES (scale: relative log) ─────────────────────────────")
# measures of model performance, not real work performance 
# training data: how well does the model perform on data it knows
train_metrics = compute_metrics_by_fold(train_df, target_col, 'prediction')
print("\n── TRAIN ──")
print(train_metrics.to_string(index=False))
# test data: how well does the model perform on data it doesn't know
test_metrics = compute_metrics_by_fold(test_df, target_col, 'prediction')
print("\n── TEST ──")
print(test_metrics.to_string(index=False))
# test data, but using median QRF model
qrf_median_metrics = compute_metrics_by_fold(test_df_qrf, target_col, "q50")
print("\n── TEST: QRF Q50 ──")
print(qrf_median_metrics.to_string(index=False))
# measure of QRF interval performance 
interval_metrics = compute_interval_metrics_by_fold(test_df_qrf, target_col, 'q10', 'q90')
print("\n── TEST: QRF interval ──")
print(interval_metrics.to_string(index=False))

# measures of model on actual variable of interest 
# log scale is more appropriate because of right-tail distribution in underlying data 
# can calculate on original scale, but would be heavily skewed by large outliers 
print("\n── PREDICTED ACTUAL INTENSITIES (scale: log) ────────────────────────────────────────")
convt_log_metrics = compute_metrics_by_fold(test_df, 'log_region_labor_intensity_jobs_per_million_tonne', 'predicted_log_labor_intensity_jobs_per_million_tonne')
print("\n── TEST ──")
print(convt_log_metrics.to_string(index=False))
# QRF interval  
interval_metrics = compute_interval_metrics_by_fold(test_df_qrf, 'log_region_labor_intensity_jobs_per_million_tonne', 'q10_predicted_log_labor_intensity_jobs_per_million_tonne', 'q90_predicted_log_labor_intensity_jobs_per_million_tonne')
print("\n── TEST: QRF interval ──")
print(interval_metrics.to_string(index=False))

# measures of naive country average model on actual variable of interest 
# log scale is more appropriate because of right-tail distribution in underlying data 
# can calculate on original scale, but would be heavily skewed by large outliers 
print("\n── COUNTRY AVERAGE MODEL (scale: log) ────────────────----------───────-─────────────")
country_log_metrics = compute_metrics_by_fold(test_df, 'log_region_labor_intensity_jobs_per_million_tonne', 'log_country_labor_intensity_jobs_per_million_tonne')
print("\n── TEST ──")
print(country_log_metrics.to_string(index=False))

### save metrics -------------------------------------------------------------------------

# output directory
out_dir = Path(f'{cd}/Results/RF_models_final/labor_rf_spatial_CV')
out_dir_qrf = Path(f'{cd}/Results/RF_models_final/labor_qrf_spatial_CV')

# relative log metrics (train + test combined)
train_metrics["split"] = "train"
test_metrics["split"]  = "test"
relative_metrics = pd.concat([train_metrics, test_metrics], ignore_index=True)
relative_metrics.to_csv(out_dir / "relative_log_metrics.csv", index=False)

# actual intensities, log scale (test only)
convt_log_metrics["split"] = "test"
convt_log_metrics.to_csv(out_dir / "actual_log_metrics.csv", index=False)

# country average baseline, log scale (test only)
country_log_metrics["split"] = "test"
country_log_metrics.to_csv(out_dir / "country_avg_log_metrics.csv", index=False)

# QRF interval metrics (coverage / width, test only)
interval_metrics["split"] = "test"
interval_metrics.to_csv(out_dir_qrf / "qrf_interval_metrics.csv", index=False)

# QRF median (q50) point-prediction metrics, relative log scale (test only)
qrf_median_metrics["split"] = "test"
qrf_median_metrics.to_csv(out_dir_qrf / "qrf_median_metrics.csv", index=False)



── PREDICTED RELATIVE INTENSITIES (scale: relative log) ─────────────────────────────

── TRAIN ──
    fold       n       R2     RMSE      MAE     Bias  Pearson_r
  fold_1  2712.0 0.825100 0.504282 0.371943 0.002872   0.918058
  fold_2  2666.0 0.821312 0.573793 0.430540 0.018556   0.919793
  fold_3  3189.0 0.854409 0.544174 0.410435 0.019619   0.933240
  fold_4  3191.0 0.855508 0.509470 0.383745 0.013636   0.935019
  fold_5  3190.0 0.855782 0.519674 0.390177 0.019469   0.934632
    mean 14948.0 0.842422 0.530279 0.397368 0.014830   0.928149
variance     NaN 0.000310 0.000827 0.000539 0.000051   0.000072

── TEST ──
    fold      n        R2     RMSE      MAE      Bias  Pearson_r
  fold_1 1025.0  0.509981 1.074342 0.865632  0.316109   0.800463
  fold_2 1071.0  0.658988 0.762938 0.607847  0.017389   0.813307
  fold_3  548.0 -0.076289 0.696047 0.529080 -0.285271   0.497095
  fold_4  546.0  0.534472 0.917121 0.710815 -0.153639   0.740210
  fold_5  547.0  0.539893 0.834541 0.629489  0.0315

/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
